# Food Delivery Data Integration – Hackathon

**Name:** Om Dilip Surve  
**Date:** 31 January 2026


In [1]:
import pandas as pd
import sqlite3

print("Imports successful")


Imports successful


In [6]:
orders = pd.read_csv("orders.csv")
orders.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [7]:
users = pd.read_json("users.json")
users.head()


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [9]:
# Create a temporary in-memory database
conn = sqlite3.connect(":memory:")


In [10]:
# Read SQL file
file = open("restaurants.sql")
sql = file.read()


In [11]:
# Execute SQL commands
conn.executescript(sql)


In [12]:
# Load restaurants table into DataFrame
restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
restaurants.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [13]:
# Merge orders with users
orders_users = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)


In [14]:
# Merge with restaurants
final_data = pd.merge(
    orders_users,
    restaurants,
    on="restaurant_id",
    how="left"
)

final_data.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [15]:
final_data.to_csv("final_food_delivery_dataset.csv", index=False)
print("Final dataset saved successfully")


Final dataset saved successfully


In [17]:
gold_orders = final_data[final_data["membership"] == "Gold"]
city_revenue = gold_orders.groupby("city")["total_amount"].sum()
city_revenue_sorted = city_revenue.sort_values(ascending=False)
city_revenue_sorted



city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [18]:
# Check column names (run once if unsure)
final_data.columns
# Calculate average order value by cuisine
cuisine_avg_order = (
    final_data
    .groupby("cuisine")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)

cuisine_avg_order
# Cuisine with highest average order value
cuisine_avg_order.idxmax()


'Mexican'

In [19]:
# Calculate total order value per user
user_total_spend = (
    final_data
    .groupby("user_id")["total_amount"]
    .sum()
)

# Count users with total spend > 1000
user_count = (user_total_spend > 1000).sum()
user_count


np.int64(2544)

In [22]:
# Create rating ranges
rating_bins = [0, 3.5, 4.0, 4.5, 5.0]
rating_labels = ["<3.5", "3.5–4.0", "4.0–4.5", "4.5–5.0"]

final_data["rating_range"] = pd.cut(
    final_data["rating"],
    bins=rating_bins,
    labels=rating_labels,
    include_lowest=True
)

# Calculate total revenue per rating range (silence FutureWarning)
rating_revenue = (
    final_data
    .groupby("rating_range", observed=False)["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

rating_revenue
# Rating range with highest total revenue
rating_revenue.idxmax()


'4.5–5.0'

In [24]:
# Filter only Gold members
gold_orders = final_data[final_data["membership"] == "Gold"]

# Calculate average order value by city
city_avg_order = (
    gold_orders
    .groupby("city")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)

city_avg_order
# City with highest average order value among Gold members
city_avg_order.idxmax()


'Chennai'

In [25]:
# Count distinct restaurants per cuisine
restaurant_count = (
    final_data
    .groupby("cuisine")["restaurant_id"]
    .nunique()
)

# Calculate total revenue per cuisine
cuisine_revenue = (
    final_data
    .groupby("cuisine")["total_amount"]
    .sum()
)

# Combine both metrics
cuisine_summary = (
    pd.DataFrame({
        "distinct_restaurants": restaurant_count,
        "total_revenue": cuisine_revenue
    })
    .sort_values(["distinct_restaurants", "total_revenue"], ascending=[True, False])
)

cuisine_summary


,distinct_restaurants,total_revenue
cuisine,,
Chinese,120,1930504.65
Italian,126,2024203.80
Indian,126,1971412.58
Mexican,128,2085503.09


In [26]:
# Total number of orders
total_orders = len(final_data)

# Number of orders placed by Gold members
gold_orders_count = len(final_data[final_data["membership"] == "Gold"])

# Percentage of orders by Gold members (rounded)
gold_order_percentage = round((gold_orders_count / total_orders) * 100)

gold_order_percentage


50

In [28]:
# Merge restaurant names into final_data (if not already present)
final_with_names = final_data.merge(
    restaurants[["restaurant_id", "restaurant_name"]],
    on="restaurant_id",
    how="left"
)

# Calculate total orders and average order value per restaurant
restaurant_stats = (
    final_with_names
    .groupby("restaurant_name")
    .agg(
        total_orders=("order_id", "count"),
        avg_order_value=("total_amount", "mean")
    )
)

# Filter restaurants with less than 20 orders
filtered = restaurant_stats[restaurant_stats["total_orders"] < 20]

# Sort by highest average order value
filtered_sorted = filtered.sort_values("avg_order_value", ascending=False)

filtered_sorted.head(1)


,total_orders,avg_order_value
restaurant_name,,
Restaurant_294,13,1040.222308


In [29]:
# Calculate total revenue by membership and cuisine
combo_revenue = (
    final_data
    .groupby(["membership", "cuisine"])["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

combo_revenue
# Combination with highest revenue
combo_revenue.idxmax()


('Regular', 'Mexican')

In [30]:
# Ensure order_date is datetime
final_data["order_date"] = pd.to_datetime(final_data["order_date"])

# Create quarter column
final_data["quarter"] = final_data["order_date"].dt.to_period("Q")

# Calculate total revenue per quarter
quarter_revenue = (
    final_data
    .groupby("quarter")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

quarter_revenue
# Quarter with highest total revenue
quarter_revenue.idxmax()


C:\Users\91960\AppData\Local\Temp\ipykernel_16636\768549166.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  final_data["order_date"] = pd.to_datetime(final_data["order_date"])


Period('2023Q3', 'Q-DEC')

In [31]:
# Count total orders placed by Gold members
gold_total_orders = len(final_data[final_data["membership"] == "Gold"])

gold_total_orders



4987

In [32]:
# Filter orders from Hyderabad city
hyderabad_orders = final_data[final_data["city"] == "Hyderabad"]

# Calculate total revenue and round to nearest integer
hyderabad_revenue = round(hyderabad_orders["total_amount"].sum())

hyderabad_revenue


1889367

In [33]:
# Count distinct users who placed at least one order
distinct_users = final_data["user_id"].nunique()

distinct_users


2883

In [34]:
# Calculate average order value for Gold members
gold_avg_order_value = round(
    final_data[final_data["membership"] == "Gold"]["total_amount"].mean(),
    2
)

gold_avg_order_value


np.float64(797.15)

In [35]:
# Count orders for restaurants with rating >= 4.5
high_rating_orders_count = len(final_data[final_data["rating"] >= 4.5])

high_rating_orders_count


3374

In [36]:
# Filter only Gold members
gold_orders = final_data[final_data["membership"] == "Gold"]

# Find total revenue by city for Gold members
gold_city_revenue = (
    gold_orders
    .groupby("city")["total_amount"]
    .sum()
)

# Identify the top revenue city among Gold members
top_city = gold_city_revenue.idxmax()

# Count number of orders in that city among Gold members
top_city_gold_orders = len(gold_orders[gold_orders["city"] == top_city])

top_city_gold_orders


1337